In [ ]:
from pathlib import Path
from torch import nn
import torch

from src import datasets
from src.data_utils import (
    FullDataStreamer,
    split_train_validation,
    calculate_class_weights,
 )

In [ ]:
import tomllib

datadir = Path('../data')
configfile = Path("config.toml")

with configfile.open('rb') as f:
    config = tomllib.load(f)
print(config)

In [ ]:
trainfile = datadir / (config['arrhythmia'] + '_train.parq')
testfile = datadir / (config['arrhythmia'] + '_test.parq')
trainfile, testfile

In [ ]:
shape = (16, 12)
full_traindataset = datasets.HeartDataset2D(trainfile, target="target", shape=shape)
testdataset = datasets.HeartDataset2D(testfile, target="target", shape=shape)
traindataset, validdataset = split_train_validation(
    full_traindataset, validation_fraction=0.2, seed=42
)
len(traindataset), len(validdataset), len(testdataset)


In [ ]:
trainstreamer = FullDataStreamer(traindataset, batch_size=32)
validstreamer = FullDataStreamer(validdataset, batch_size=32, shuffle=False)
teststreamer = FullDataStreamer(testdataset, batch_size=32, shuffle=False)
len(trainstreamer), len(validstreamer), len(teststreamer)

In [ ]:
# ============================================
# Experiment 1 - Baseline 2D CNN 
# ============================================

In [ ]:
trainloader = trainstreamer.stream()

x, y = next(iter(trainloader))

print(x.shape)
print(y.shape)

In [ ]:
from src.helpers import run_experiment
from src.networks import CNN2D


run_experiment(
    model_function=lambda: CNN2D(
        num_classes=5,
        filters=16,
        matrixshape=(16, 12),
    ),
    experiment_name="2D CNN Baseline",
    trainstreamer=trainstreamer,
    validstreamer=validstreamer,
    teststreamer=teststreamer,
    loss_fn=nn.CrossEntropyLoss(),
    class_weights=True,
    parallel=False,
    runs=3,
    epochs=5,
    filename="results/results.csv",
)

In [ ]:
# ============================================
# Experiment 2 - 2D CNN met class weights
# ============================================

In [ ]:
# Class weights uitsluitend uit de trainingssubset berekenen
class_weights = calculate_class_weights(traindataset)
classes = ["N", "S", "V", "F", "Q"]
for cls, weight in zip(classes, class_weights):
    print(f"{cls}: weight={weight:.3f}")


In [ ]:
run_experiment(
    model_function=lambda: CNN2D(
        num_classes=5,
        filters=16,
        matrixshape=(16, 12),
    ),
    experiment_name="2D CNN Weighted",
    trainstreamer=trainstreamer,
    validstreamer=validstreamer,
    teststreamer=teststreamer,
    loss_fn=nn.CrossEntropyLoss(
        weight=class_weights,
    ),
    class_weights=True,
    parallel=False,
    runs=3,
    epochs=5,
    filename="results/results.csv",
)

In [ ]:
# =============================================================
# Experiment 3 - 2D CNN met class weights en paralelle kernels
# =============================================================

In [ ]:
from src.networks import ParallelCNN2D


run_experiment(
    model_function=lambda: ParallelCNN2D(
        num_classes=5,
        filters=16,
        matrixshape=(16, 12),
    ),
    experiment_name="Parallelle 2D CNN",
    trainstreamer=trainstreamer,
    validstreamer=validstreamer,
    teststreamer=teststreamer,
    loss_fn=nn.CrossEntropyLoss(
        weight=class_weights,
    ),
    class_weights=True,
    parallel=True,
    runs=3,
    epochs=5,
    filename="results/results.csv",
)